In [10]:
# ================================================================================
# PhyloCNN testing and conformal calibration under the BDEIVH model
# ================================================================================
#
# This notebook loads a previously trained PhyloCNN model, generates predictions
# for the calibration and test datasets, applies conformalized quantile
# regression (CQR), and exports the test results.
# ================================================================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import keras

from keras.saving import register_keras_serializable
from keras.models import model_from_json

from tensorflow.keras.models import model_from_json
from keras.src.saving import serialization_lib

QUANTILES = (
    0.025,  # lower
    0.5,    # median
    0.975   # upper
)


# Number of tip range corresponding to the training trees
TREE_SIZE = '30_50'

TREE_TYPE = 'TT' # transmission trees
# TREE_TYPE = 'OROV' # OROV-like trees

# Path to the test tree parameter csv file
TEST_PARAMS = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_test', TREE_SIZE, 'trees.log')

# Path to the tsv-encoded test tree file
TEST_ENC_TREES = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_test', TREE_SIZE, 'trees.tsv.xz')


# Path where the estimated parameters will be saved
TEST_EST = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_test', TREE_SIZE, 'estimates_bdeivh.csv')

# Path where the true parameters will be saved
TEST_TRUE = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_test', TREE_SIZE, 'target_bdeivh.csv')


# Path to the calibration tree parameter csv file
CALIBRATION_PARAMS = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_calibrate', TREE_SIZE, 'trees.log')

# Path to the tsv-encoded calibration tree file
CALIBRATION_ENC_TREES = os.path.join('..', 'tree_simulation', f'{TREE_TYPE}_calibrate', TREE_SIZE, 'trees.tsv.xz')
                               
# Path to the folder where the trained model is be stored
MODEL_PATH = os.path.join('..', 'bdeivh', 'models')
                               
# Model name -- will be used for looking for the model in the MODEL_PATH folder as two files: .json and .weights.h5
MODEL_NAME = f'{TREE_SIZE}_{TREE_TYPE}_BDEIVH'


In [11]:
# Load the test and calibration parameters and tree encodings.

param_test = pd.read_csv(TEST_PARAMS, sep=',')

param_calibration = pd.read_csv(CALIBRATION_PARAMS, sep=',')

encoding_test = pd.read_csv(
    TEST_ENC_TREES,
    sep="\t",
    header=0,
    index_col=0
).values.astype('float32').reshape(-1, 800, 11)

encoding_calibration = pd.read_csv(
    CALIBRATION_ENC_TREES,
    sep="\t",
    header=0,
    index_col=0
).values.astype('float32').reshape(-1, 800, 11)

# Match parameter rows to encoding rows.
param_test = param_test.iloc[:encoding_test.shape[0]].reset_index(drop=True)
param_calibration = param_calibration.iloc[
    :encoding_calibration.shape[0]
].reset_index(drop=True)

print(f"Test: {encoding_test.shape[0]} trees | {param_test.shape[0]} parameter rows")
print(
    f"Calibration: {encoding_calibration.shape[0]} trees | "
    f"{param_calibration.shape[0]} parameter rows"
)

Test: 10 trees | 10 parameter rows
Calibration: 20 trees | 20 parameter rows


In [12]:
# Define targets and prepare the test and calibration labels.

target_1 = "Rvh"        # Transmission from vectors to humans
target_2 = "Rhv"        # Transmission from humans to vectors
target_3 = "Rhh"        # Transmission among humans
target_4 = "d_v"        # Total duration of the infection in vectors
target_5 = "d_h"        # Total duration of the infection in humans
target_6 = "v2h_ratio"  # Ratio of vectors and humans

target_cols = [target_1, target_2, target_3, target_4, target_5, target_6]

# Extract the tree-specific rescaling factors.
param_test['resc_factor'] = encoding_test[:, -1, -1]
param_calibration['resc_factor'] = encoding_calibration[:, -1, -1]

# Rescale time-dependent targets.
for key in (target_4, target_5):
    param_test[key] = param_test[key] / param_test['resc_factor']
    param_calibration[key] = (
        param_calibration[key] / param_calibration['resc_factor']
    )

# Remove the final row containing the rescaling factor.
encoding_test = np.delete(encoding_test, -1, axis=1)
encoding_calibration = np.delete(encoding_calibration, -1, axis=1)

Y_test = pd.DataFrame(param_test[target_cols]).reset_index(drop=True)
Y_calibration = pd.DataFrame(
    param_calibration[target_cols]
).reset_index(drop=True)

print("Y test:", Y_test.shape)
print("Y calibration:", Y_calibration.shape)


Y test: (10, 6)
Y calibration: (20, 6)


In [13]:
# Add the tree-level sampling probability (p) as an additional feature.

n_rows_test = encoding_test.shape[1]
n_rows_calibration = encoding_calibration.shape[1]

samp_proba_list_test = np.asarray(param_test['p'])
encoding_test = np.concatenate(
    (
        encoding_test,
        np.repeat(samp_proba_list_test, n_rows_test).reshape(
            -1, n_rows_test, 1
        )
    ),
    axis=2
)

samp_proba_list_calibration = np.asarray(param_calibration['p'])
encoding_calibration = np.concatenate(
    (
        encoding_calibration,
        np.repeat(
            samp_proba_list_calibration, n_rows_calibration
        ).reshape(-1, n_rows_calibration, 1)
    ),
    axis=2
)

print("Test encoding:", encoding_test.shape)
print("Calibration encoding:", encoding_calibration.shape)


Test encoding: (10, 799, 12)
Calibration encoding: (20, 799, 12)


In [14]:
# This function converts each encoded tree into two fixed-length channels:
# one for leaves and one for internal nodes.
#
# Leaves and internal nodes are sorted by distance from the root and
# zero-padded to a maximum of 400 rows per channel.

def encode_pad_0s_rootage_single(enc, max_len=400):
    enc_pad = []

    for i in range(enc.shape[0]):
        # Leaves: column 3 == 1.
        leaves = enc[i][enc[i, :, 3] == 1]
        leaves = leaves[np.argsort(leaves[:, 1])]
        leaves = np.pad(leaves, [(0, (max_len - leaves.shape[0])), (0, 0)], mode='constant')

        # Internal nodes: column 3 > 1.
        nodes = enc[i][enc[i, :, 3] > 1]
        nodes = nodes[np.argsort(nodes[:, 1])]

        # Copy the last node's value to balance leaves and nodes.
        # This follows the original notebook behavior.
        nodes = np.append(nodes, nodes[-1].reshape(1, -1), axis=0)
        nodes = np.pad(nodes, [(0, (max_len - nodes.shape[0])), (0, 0)], mode='constant')

        enc_pad.append(np.stack((leaves, nodes), axis=2))

    return np.array(enc_pad)

# Change encoding to order by root age and pad with 0s.
encoding_pad_test = encode_pad_0s_rootage_single(encoding_test)
encoding_pad_calibration = encode_pad_0s_rootage_single(encoding_calibration)

# Verify compatibility with the model input shape: (400, 12, 2).
print("Test encoding:", encoding_pad_test.shape)
print("Calibration encoding:", encoding_pad_calibration.shape)

Test encoding: (10, 400, 12, 2)
Calibration encoding: (20, 400, 12, 2)


In [15]:
# Load the model architecture and trained weights.
serialization_lib.enable_unsafe_deserialization()

MODEL_JSON = os.path.join(MODEL_PATH, f'{MODEL_NAME}.json')

MODEL_WEIGHTS = os.path.join(MODEL_PATH, f'{MODEL_NAME}.weights.h5')

# Necessário para desserializar a Lambda layer do modelo confiável.
serialization_lib.enable_unsafe_deserialization()

with open(MODEL_JSON, "r", encoding="utf-8") as json_file:
    estimator = model_from_json(json_file.read())

estimator.load_weights(MODEL_WEIGHTS)

print("Model loaded successfully.")
print("Model input shape:", estimator.input_shape)


Model loaded successfully.
Model input shape: (None, 400, 12, 2)


In [16]:
# Generate predictions for the test and calibration sets.
Y_pred_test = estimator.predict(encoding_pad_test)
Y_pred_calibration = estimator.predict(encoding_pad_calibration)

# Reshape predictions to:
# (n_samples, n_targets, n_quantiles)
Y_pred_test = Y_pred_test.reshape(
    -1,
    len(target_cols),
    len(QUANTILES)
)

Y_pred_calibration = Y_pred_calibration.reshape(
    -1,
    len(target_cols),
    len(QUANTILES)
)

# Column names for lower, central and upper predictions.
pred_cols = []

for col in target_cols:
    pred_cols.extend([
        f"{col}_lower",
        col,
        f"{col}_upper"
    ])

# Convert test predictions to a DataFrame.
predicted_test = pd.DataFrame(
    Y_pred_test.reshape(Y_pred_test.shape[0], -1),
    columns=pred_cols
)

predicted_test.index = Y_test.index

# Convert calibration predictions to a DataFrame.
predicted_calibration = pd.DataFrame(
    Y_pred_calibration.reshape(Y_pred_calibration.shape[0], -1),
    columns=pred_cols
)

predicted_calibration.index = Y_calibration.index

# Verify dimensions.
print("Test predictions:", predicted_test.shape)
print("Calibration predictions:", predicted_calibration.shape)

1/1 [==============================] - 0s 23ms/step
Test predictions: (10, 18)
Calibration predictions: (20, 18)


2026-11-23 00:35:25.860389: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x72e274008150 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-11-23 00:35:25.860414: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version
2026-11-23 00:35:25.866150: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [17]:
# Predict quantiles for calibration and test sets, then apply CQR.

ALPHA = 0.05  # 95% prediction intervals.

print("Calibration prediction shape:", Y_pred_calibration.shape)
print("Test prediction shape:", Y_pred_test.shape)

n_targets = len(target_cols)
n_quantiles = len(QUANTILES)

# Reshape from (n_samples, n_targets * n_quantiles) to (n_samples, n_targets, n_quantiles).
Y_pred_cal_3d = Y_pred_calibration.reshape(-1, n_targets, n_quantiles)
Y_pred_test_3d = Y_pred_test.reshape(-1, n_targets, n_quantiles)

# Quantile positions: lower, median, upper.
lower_idx = 0
median_idx = 1
upper_idx = 2

# CQR conformity score for each calibration example and target:
# score = max(q_lower - y_true, y_true - q_upper)
y_cal_values = Y_calibration[target_cols].values
cal_scores = np.maximum(
    Y_pred_cal_3d[:, :, lower_idx] - y_cal_values,
    y_cal_values - Y_pred_cal_3d[:, :, upper_idx]
)

# Split conformal finite-sample correction.
# This is computed separately for each target parameter.
n_cal = cal_scores.shape[0]
q_level = np.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
q_level = min(q_level, 1.0)

try:
    q_hat = np.quantile(cal_scores, q_level, axis=0, method='higher')
except TypeError:
    # For older NumPy versions.
    q_hat = np.quantile(cal_scores, q_level, axis=0, interpolation='higher')

print("CQR q_level:", q_level)
print("CQR q_hat per target:")
for col, q in zip(target_cols, q_hat):
    print(f"  {col}: {q}")

# Apply CQR correction to test-set intervals.
Y_pred_test_calibrated = Y_pred_test_3d.copy()
Y_pred_test_calibrated[:, :, lower_idx] = Y_pred_test_3d[:, :, lower_idx] - q_hat
Y_pred_test_calibrated[:, :, upper_idx] = Y_pred_test_3d[:, :, upper_idx] + q_hat

# Convert calibrated predictions to dataframe.
pred_cols = []
for col in target_cols:
    pred_cols.extend([
        f"{col}_lower",
        col,
        f"{col}_upper"
    ])

predicted_test = pd.DataFrame(
    Y_pred_test_calibrated.reshape(Y_pred_test_calibrated.shape[0], -1),
    columns=pred_cols
)
predicted_test.index = Y_test.index

# Optional quick empirical coverage on the held-out test set, in the scaled space.
coverage = {}
for col in target_cols:
    coverage[col] = np.mean(
        (Y_test[col].values >= predicted_test[f"{col}_lower"].values) &
        (Y_test[col].values <= predicted_test[f"{col}_upper"].values)
    )

print("Empirical test coverage after CQR, scaled space:")
for col, cov in coverage.items():
    print(f"  {col}: {cov:.3f}")

predicted_test.head()

Calibration prediction shape: (20, 6, 3)
Test prediction shape: (10, 6, 3)
CQR q_level: 1.0
CQR q_hat per target:
  Rvh: -0.029727158220487215
  Rhv: -0.1232895741030452
  Rhh: -0.19817020145303732
  d_v: 0.1829512966143207
  d_h: -0.0091367306900062
  v2h_ratio: 0.11218446284706518
Empirical test coverage after CQR, scaled space:
  Rvh: 1.000
  Rhv: 0.800
  Rhh: 0.900
  d_v: 0.900
  d_h: 1.000
  v2h_ratio: 1.000


,Rvh_lower,Rvh,Rvh_upper,Rhv_lower,Rhv,Rhv_upper,Rhh_lower,Rhh,Rhh_upper,d_v_lower,d_v,d_v_upper,d_h_lower,d_h,d_h_upper,v2h_ratio_lower,v2h_ratio,v2h_ratio_upper
0,1.552184,3.840314,6.572023,0.611780,0.589103,1.112221,1.478770,2.207837,3.729784,0.973076,2.468076,4.951729,0.763928,1.062083,1.666197,0.348099,0.995833,1.914909
1,0.748186,2.522120,5.877920,0.667540,0.985134,2.879764,1.691461,2.500756,4.045666,0.570977,1.464512,3.027916,0.345512,0.576270,1.024821,0.443375,1.522255,5.010932
2,1.113745,3.668591,7.936501,0.669345,0.991319,3.115544,2.190939,3.713370,5.038717,0.733917,1.596939,3.287175,0.502024,0.811920,1.333700,0.287461,1.028208,3.847285
3,1.253650,3.434352,6.977466,0.628398,0.752789,1.793606,1.545501,2.687514,4.147520,0.746657,1.918952,3.815967,0.826099,1.327825,1.928386,0.170055,0.669685,1.927880
4,0.555238,2.533595,7.015439,0.705241,1.492442,4.797082,2.185117,3.454232,4.739223,0.345421,0.854526,1.832250,0.282139,0.528580,0.914032,0.264541,1.312386,5.860045


In [18]:
# Restore the original scale of the time-dependent parameters
# and save the true and predicted values to CSV files.
#
# During preprocessing, the parameters d_v and d_h were divided by
# the tree-specific rescaling factor so that they matched the scale
# of the normalized phylogenetic trees used for training.
#
# Before exporting the results, these parameters are multiplied by
# the same rescaling factor to recover their original units.
#
# The remaining parameters were not rescaled during preprocessing
# and therefore require no further transformation.

import os

# Parameters that require inverse rescaling.
cols_to_rescale = [target_4, target_5]

# Tree-specific rescaling factor.
factor = param_test["resc_factor"].to_numpy(dtype=np.float32)

# Create independent copies so that the original DataFrames
# remain unchanged.
Y_test_rescaled = Y_test.copy()
predicted_test_rescaled = predicted_test.copy()

# Restore the original scale of the time-dependent parameters.
for col in cols_to_rescale:

    # Restore the true parameter values.
    Y_test_rescaled[col] = (
        Y_test_rescaled[col].to_numpy() * factor
    )

    # Restore the predicted median.
    predicted_test_rescaled[col] = (
        predicted_test_rescaled[col].to_numpy() * factor
    )

    # Restore the lower prediction limit.
    predicted_test_rescaled[f"{col}_lower"] = (
        predicted_test_rescaled[f"{col}_lower"].to_numpy() * factor
    )

    # Restore the upper prediction limit.
    predicted_test_rescaled[f"{col}_upper"] = (
        predicted_test_rescaled[f"{col}_upper"].to_numpy() * factor
    )

# Save the true parameter values.
Y_test_rescaled.to_csv(
    TEST_TRUE,
    header=True,
    index=True
)

# Save the predicted median and prediction intervals.
predicted_test_rescaled.to_csv(
    TEST_EST,
    header=True,
    index=True
)

# Confirm that the files were successfully written.
print(f"Prediction files saved to {TEST_EST}")

Prediction files saved to ../tree_simulation/TT_test/30_50/estimates_bdeivh.csv
